In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import webbrowser
import os

In [2]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\VINEETHA\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [3]:
apps_df=pd.read_csv('Play Store Data.csv')
reviews_df=pd.read_csv('User Reviews.csv')

In [4]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [5]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [6]:
#pd.read_csv() : csv files
#pd.read_excel() : excel files
#pd.read_sql() : SQL Databases
#pd.read_json() : JSON Files

In [7]:
#df.isnull() : Missing values
#df.dropna() : Removes rows and columns that contain the missing values
#df.fillna() : Fills missing values

In [8]:
#df.duplicated() : Identifies duplicates
#df.drop_duplicates() : Removes duplicate rows

In [9]:
apps_df = apps_df.dropna(subset=['Rating'])
apps_df = apps_df.fillna(apps_df.mode().iloc[0])
apps_df = apps_df.drop_duplicates()
apps_df = apps_df[apps_df['Rating'] <= 5]
reviews_df = reviews_df.dropna(subset=['Translated_Review'])

In [10]:
apps_df.dtypes

App                   str
Category              str
Rating            float64
Reviews               str
Size                  str
Installs              str
Type                  str
Price                 str
Content Rating        str
Genres                str
Last Updated          str
Current Ver           str
Android Ver           str
dtype: object

In [11]:
#Convert the Installs columns to numeric by removing commas and +
apps_df['Installs']=apps_df['Installs'].str.replace(',','').str.replace('+','').astype(int)

#Convert Price column to numeric after removing $
apps_df['Price']=apps_df['Price'].str.replace('$','').astype(float)

In [12]:
apps_df.dtypes

App                   str
Category              str
Rating            float64
Reviews               str
Size                  str
Installs            int64
Type                  str
Price             float64
Content Rating        str
Genres                str
Last Updated          str
Current Ver           str
Android Ver           str
dtype: object

In [13]:
merged_df=pd.merge(apps_df,reviews_df,on='App',how='inner')

In [14]:
merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I love colors inspyering,Positive,0.500,0.600000
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I hate,Negative,-0.800,0.900000


In [15]:
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M',''))
    elif 'k' in size:
        return float(size.replace('k',''))/1024
    else:
        return np.nan
apps_df['Size']=apps_df['Size'].apply(convert_size)

In [16]:
apps_df

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10834,FR Calculator,FAMILY,4.0,7,2.6,500,Free,0.0,Everyone,Education,"June 18, 2017",1.0.0,4.1 and up
10836,Sya9a Maroc - FR,FAMILY,4.5,38,53.0,5000,Free,0.0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10837,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3.6,100,Free,0.0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10839,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,NaN,1000,Free,0.0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device


In [17]:
#Lograrithmic
apps_df['Log_Installs']=np.log(apps_df['Installs'])

In [18]:
apps_df['Reviews']=apps_df['Reviews'].astype(int)

In [19]:
apps_df['Log_Reviews']=np.log(apps_df['Reviews'])

In [20]:
apps_df.dtypes

App                   str
Category              str
Rating            float64
Reviews             int64
Size              float64
Installs            int64
Type                  str
Price             float64
Content Rating        str
Genres                str
Last Updated          str
Current Ver           str
Android Ver           str
Log_Installs      float64
Log_Reviews       float64
dtype: object

In [21]:
def rating_group(rating):
    if rating >= 4:
        return 'Top rated app'
    elif rating >=3:
        return 'Above average'
    elif rating >=2:
        return 'Average'
    else:
        return 'Below Average'
apps_df['Rating_Group']=apps_df['Rating'].apply(rating_group)

In [22]:
#Revenue column
apps_df['Revenue']=apps_df['Price']*apps_df['Installs']

In [23]:
sia = SentimentIntensityAnalyzer()

In [24]:
#Polarity Scores in SIA
#Positive, Negative, Neutral and Compound: -1 - Very negative ; +1 - Very positive

In [25]:
review = "This app is amazing! I love the new features."
sentiment_score= sia.polarity_scores(review)
print(sentiment_score)

{'neg': 0.0, 'neu': 0.42, 'pos': 0.58, 'compound': 0.8516}


In [26]:
review = "This app is very bad! I hate the new features."
sentiment_score= sia.polarity_scores(review)
print(sentiment_score)

{'neg': 0.535, 'neu': 0.465, 'pos': 0.0, 'compound': -0.8427}


In [27]:
review = "This app is okay."
sentiment_score= sia.polarity_scores(review)
print(sentiment_score)

{'neg': 0.0, 'neu': 0.612, 'pos': 0.388, 'compound': 0.2263}


In [28]:
reviews_df['Sentiment_Score']=reviews_df['Translated_Review'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

In [29]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Sentiment_Score
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333,0.9531
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462,0.6597
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000,0.6249
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000,0.6369
5,10 Best Foods for You,Best way,Positive,1.00,0.300000,0.6369


In [30]:
apps_df['Last Updated']=pd.to_datetime(apps_df['Last Updated'],errors='coerce')

In [31]:
apps_df['Year']=apps_df['Last Updated'].dt.year

In [32]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue,Year
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,9.210340,5.068904,Top rated app,0.0,2018
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,13.122363,6.874198,Above average,0.0,2018
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,15.424948,11.379508,Top rated app,0.0,2018
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,17.727534,12.281384,Top rated app,0.0,2018
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,11.512925,6.874198,Top rated app,0.0,2018


In [33]:
html_files_path="./"
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

In [34]:
plot_containers = ""

In [35]:
# Save each Plotly figure to an HTML file
def save_plot_as_html(fig, filename, insight):
    global plot_containers
    filepath = os.path.join(html_files_path, filename)
    html_content = pio.to_html(fig, full_html=False, include_plotlyjs='inline')
    # Append the plot and its insight to plot_containers
    plot_containers += f"""
    <div class="plot-container" id="{filename}" onclick="openPlot('{filename}')">
        <div class="plot">{html_content}</div>
        <div class="insights">{insight}</div>
    </div>
    """
    fig.write_html(filepath, full_html=False, include_plotlyjs='inline')

In [36]:
plot_width=400
plot_height=300
plot_bg_color='black'
text_color='white'
title_font={'size':16}
axis_font={'size':12}

In [37]:
#Figure 1
category_counts=apps_df['Category'].value_counts().nlargest(10)
fig1=px.bar(
    x=category_counts.index,
    y=category_counts.values,
    labels={'x':'Category','y':'Count'},
    title='Top Categories on Play Store',
    color=category_counts.index,
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=400,
    height=300
)
fig1.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig1,"Category Graph 1.html","The top categories on the Play Store are dominated by tools, entertainment, and productivity apps")
            

In [38]:
#Figure 2
type_counts=apps_df['Type'].value_counts()
fig2=px.pie(
    values=type_counts.values,
    names=type_counts.index,
    title='App Type Distribution',
    color_discrete_sequence=px.colors.sequential.RdBu,
    width=400,
    height=300
)
fig2.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig2,"Type Graph 2.html","Most apps on the Playstore are free, indicating a strategy to attract users first and monetize through ads or in app purchases")

In [39]:
#Figure 3
fig3=px.histogram(
    apps_df,
    x='Rating',
    nbins=20,
    title='Rating Distribution',
    color_discrete_sequence=['#636EFA'],
    width=400,
    height=300
)
fig3.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig3,"Rating Graph 3.html","Ratings are skewed towards higher values, suggesting that most apps are rated favorably by users")

In [40]:
#Figure 4
sentiment_counts=reviews_df['Sentiment_Score'].value_counts()
fig4=px.bar(
    x=sentiment_counts.index,
    y=sentiment_counts.values,
    labels={'x':'Sentiment Score','y':'Count'},
    title='Sentiment Distribution',
    color=sentiment_counts.index,
    color_discrete_sequence=px.colors.sequential.RdPu,
    width=400,
    height=300
)
fig4.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig4,"Sentiment Graph 4.html","Sentiments in reviews show a mix of positive and negative feedback, with a slight lean towards positive sentiments")

In [41]:
#Figure 5
installs_by_category=apps_df.groupby('Category')['Installs'].sum().nlargest(10)
fig5=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    orientation='h',
    labels={'x':'Installs','y':'Category'},
    title='Installs by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Blues,
    width=400,
    height=300
)
fig5.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig5,"Installs Graph 5.html","The categories with the most installs are social and communication apps, reflecting their broad appeal and daily usage")

In [42]:
# Updates Per Year Plot
updates_per_year = apps_df['Last Updated'].dt.year.value_counts().sort_index()
fig6 = px.line(
    x=updates_per_year.index,
    y=updates_per_year.values,
    labels={'x': 'Year', 'y': 'Number of Updates'},
    title='Number of Updates Over the Years',
    color_discrete_sequence=['#AB63FA'],
    width=plot_width,
    height=plot_height
)
fig6.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig6, "Updates Graph 6.html", "Updates have been increasing over the years, showing that developers are actively maintaining and improving their apps.")

In [43]:
#Figure 7
revenue_by_category=apps_df.groupby('Category')['Revenue'].sum().nlargest(10)
fig7=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    labels={'x':'Category','y':'Revenue'},
    title='Revenue by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Greens,
    width=400,
    height=300
)
fig7.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig7,"Revenue Graph 7.html","Categories such as Business and Productivity lead in revenue generation, indicating their monetization potential")

In [44]:
#Figure 8
genre_counts=apps_df['Genres'].str.split(';',expand=True).stack().value_counts().nlargest(10)
fig8=px.bar(
    x=genre_counts.index,
    y=genre_counts.values,
    labels={'x':'Genre','y':'Count'},
    title='Top Genres',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.OrRd,
    width=400,
    height=300
)
fig8.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig8,"Genre Graph 8.html","Action and Casual genres are the most common, reflecting users' preference for engaging and easy-to-play games")

In [45]:
#Figure 9
fig9=px.scatter(
    apps_df,
    x='Last Updated',
    y='Rating',
    color='Type',
    title='Impact of Last Update on Rating',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    width=400,
    height=300
)
fig9.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig9,"Update Graph 9.html","The Scatter Plot shows a weak correlation between the last update and ratings, suggesting that more frequent updates dont always result in better ratings.")

In [46]:
#Figure 10
fig10=px.box(
    apps_df,
    x='Type',
    y='Rating',
    color='Type',
    title='Rating for Paid vs Free Apps',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=400,
    height=300
)
fig10.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig10,"Paid Free Graph 10.html","Paid apps generally have higher ratings compared to free apps, suggesting that users expect higher quality from apps they pay for")

In [47]:
print("========== INTERNSHIP TASKS ==========")

========== INTERNSHIP TASKS ==========


In [48]:
def convert_size(size):
    if pd.isna(size):
        return None
    size = str(size)

    if 'M' in size:
        return float(size.replace('M', ''))

    elif 'k' in size:
        return float(size.replace('k', '')) / 1024

    return None

In [49]:
merged_df["Rating"] = pd.to_numeric(
    merged_df["Rating"],
    errors="coerce"
)

merged_df["Reviews"] = pd.to_numeric(
    merged_df["Reviews"],
    errors="coerce"
)

merged_df["Installs"] = pd.to_numeric(
    merged_df["Installs"],
    errors="coerce"
)

merged_df["Sentiment_Subjectivity"] = pd.to_numeric(
    merged_df["Sentiment_Subjectivity"],
    errors="coerce"
)

print("Numeric conversion completed")

Numeric conversion completed


In [50]:
print(merged_df["Reviews"].dtype)
print(merged_df["Rating"].dtype)
print(merged_df["Installs"].dtype)

int64
float64
int64


In [51]:
translation_map = {
    "BEAUTY": "सौंदर्य",
    "BUSINESS": "வணிகம்",
    "DATING": "Partnersuche",
    "TRAVEL_AND_LOCAL": "Voyage et Local",
    "PRODUCTIVITY": "Productividad",
    "PHOTOGRAPHY": "写真"
}

merged_df["Category_Display"] = merged_df["Category"].replace(translation_map)

In [52]:
merged_df["Size_MB"] = merged_df["Size"].apply(convert_size)

In [53]:
merged_df["Installs"] = (
    merged_df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

merged_df["Installs"] = pd.to_numeric(
    merged_df["Installs"],
    errors="coerce"
)

In [54]:
merged_df = merged_df.dropna(
    subset=["Size_MB", "Installs", "Rating"]
)

In [55]:
from datetime import datetime
import pytz

def is_time_allowed(start_hour, end_hour):

    ist = pytz.timezone("Asia/Kolkata")

    current_hour = datetime.now(ist).hour

    return start_hour <= current_hour < end_hour

In [56]:
if is_time_allowed(17,19):

    categories_needed = [
        "GAME",
        "BEAUTY",
        "BUSINESS",
        "COMICS",
        "COMMUNICATION",
        "DATING",
        "ENTERTAINMENT",
        "SOCIAL",
        "EVENTS"
    ]

    df1 = merged_df[
        (merged_df["Rating"] > 3.5) &
        (merged_df["Reviews"] > 500) &
        (merged_df["Installs"] > 50000) &
        (merged_df["Sentiment_Subjectivity"] > 0.5) &
        (~merged_df["App"].str.contains("S", case=False, na=False)) &
        (merged_df["Category"].isin(categories_needed))
    ].copy()

    df1["Category_Display"] = df1["Category"].replace({
        "BEAUTY": "सौंदर्य",
        "BUSINESS": "வணிகம்",
        "DATING": "Partnersuche"
    })

    task_fig1 = px.scatter(
        df1,
        x="Size_MB",
        y="Rating",
        size="Installs",
        color="Category_Display",
        hover_name="App",
        title="Task 1 Bubble Chart",
        color_discrete_map={"GAME": "pink"}
    )

    task_fig1.update_layout(
    width=400,
    height=300
)

    task_fig1.show()
    save_plot_as_html(
    task_fig1,
    "Internship_Task_1.html",
    "Bubble chart analysis"
)


else:
    plot_containers += """
    <div class="plot-container">
        Available only between 5 PM and 7 PM IST
    </div>
    """

In [57]:
if is_time_allowed(18,20):

    df2 = merged_df.groupby("Category")["Installs"].sum().reset_index()

    df2 = df2[
        ~df2["Category"].str.startswith(
            ("A","C","G","S"),
            na=False
        )
    ]

    df2 = df2.sort_values(
        by="Installs",
        ascending=False
    ).head(5)

    df2["Highlight"] = np.where(
        df2["Installs"] > 1000000,
        "Above 1M",
        "Below 1M"
    )

    countries = ["IND","USA","BRA","DEU","CAN"]

    df2["Country"] = countries[:len(df2)]

    task_fig2 = px.choropleth(
        df2,
        locations="Country",
        locationmode="ISO-3",
        color="Installs",
        hover_name="Category",
        hover_data=["Highlight"],
        color_continuous_scale="Plasma",
        title="Task 2 Global Installs by Category"
    )

    task_fig2.update_layout(
    width=400,
    height=300
)

    task_fig2.show()
    save_plot_as_html(
    task_fig2,
    "Internship_Task_2.html",
    "Choropleth map analysis"
)

else:
    plot_containers += """
    <div class="plot-container">
        Available only between 6 PM and 8 PM IST
    </div>
    """

In [58]:
merged_df["Reviews"] = pd.to_numeric(
    merged_df["Reviews"],
    errors="coerce"
)

merged_df["Rating"] = pd.to_numeric(
    merged_df["Rating"],
    errors="coerce"
)

merged_df["Installs"] = pd.to_numeric(
    merged_df["Installs"],
    errors="coerce"
)

merged_df["Sentiment_Subjectivity"] = pd.to_numeric(
    merged_df["Sentiment_Subjectivity"],
    errors="coerce"
)

print(merged_df["Reviews"].dtype)
print(merged_df["Rating"].dtype)
print(merged_df["Installs"].dtype)
print(merged_df["Sentiment_Subjectivity"].dtype)

int64
float64
int64
float64


In [59]:
print(merged_df["Reviews"].dtype)

int64


In [60]:
if is_time_allowed(18,21):

    df3 = merged_df[
        (merged_df["Reviews"] > 500) &
        (~merged_df["App"].str.startswith(
            ("X","Y","Z"),
            na=False
        )) &
        (~merged_df["App"].str.contains(
            "S",
            case=False,
            na=False
        )) &
        (merged_df["Category"].str.startswith(
            ("E","C","B"),
            na=False
        ))
    ].copy()

    df3["Category_Display"] = df3["Category"].replace({
        "BEAUTY": "सौंदर्य",
        "BUSINESS": "வணிகம்",
        "DATING": "Partnersuche"
    })

    df3["Month"] = pd.to_datetime(
        df3["Last Updated"],
        errors="coerce"
    ).dt.to_period("M").astype(str)

    trend = df3.groupby(
        ["Month","Category_Display"]
    )["Installs"].sum().reset_index()

    task_fig3 = px.line(
        trend,
        x="Month",
        y="Installs",
        color="Category_Display",
        title="Task 3 Installs Trend Over Time"
    )

    task_fig3.update_layout(
    width=400,
    height=300
)

    task_fig3.show()
    save_plot_as_html(
    task_fig3,
    "Internship_Task_3.html",
    "Task 3 analysis"
)
    

else:
     plot_containers += """
    <div class="plot-container">
        Available only between 6 PM and 9 PM IST
    </div>
    """

In [61]:
print(merged_df.dtypes[["Reviews","Rating","Installs","Sentiment_Subjectivity"]])

Reviews                     int64
Rating                    float64
Installs                    int64
Sentiment_Subjectivity    float64
dtype: object


In [62]:
merged_df["Reviews"] = pd.to_numeric(
    merged_df["Reviews"].astype(str).str.replace(",", ""),
    errors="coerce"
)

merged_df["Rating"] = pd.to_numeric(
    merged_df["Rating"],
    errors="coerce"
)

merged_df["Installs"] = pd.to_numeric(
    merged_df["Installs"],
    errors="coerce"
)

print(merged_df["Reviews"].dtype)

int64


In [63]:
if is_time_allowed(16,18):

    df4 = merged_df[
    (merged_df["Rating"] >= 4.2) &
    (merged_df["Reviews"] > 1000) &
    (merged_df["Size_MB"].between(20,80)) &
    (~merged_df["App"].str.contains(r"\d", regex=True, na=False)) &
    (merged_df["Category"].str.startswith(("T","P"), na=False))
].copy()

    df4["Category_Display"] = df4["Category"].replace({
        "TRAVEL_AND_LOCAL":"Voyage et Local",
        "PRODUCTIVITY":"Productividad",
        "PHOTOGRAPHY":"写真"
    })

    df4["Month"] = pd.to_datetime(
        df4["Last Updated"],
        errors="coerce"
    ).dt.to_period("M").astype(str)

    area_data = df4.groupby(
        ["Month","Category_Display"]
    )["Installs"].sum().reset_index()

    task_fig4 = px.area(
        area_data,
        x="Month",
        y="Installs",
        color="Category_Display",
        title="Task 4 Cumulative Installs"
    )

    task_fig4.update_layout(
    width=400,
    height=300
)

    task_fig4.show()
    save_plot_as_html(
    task_fig4,
    "Internship_Task_4.html",
    "Task 4 analysis"
)
    

else:
    plot_containers += """
    <div class="plot-container">
        Available only between 4 PM and 6 PM IST
    </div>
    """

In [64]:
if is_time_allowed(15,17):

    merged_df["Last Updated"] = pd.to_datetime(
        merged_df["Last Updated"],
        errors="coerce"
    )

    df5 = merged_df[
        (merged_df["Rating"] >= 4.0) &
        (merged_df["Size_MB"] >= 10) &
        (merged_df["Last Updated"].dt.month == 1)
    ]

    top_categories = df5.groupby(
        "Category"
    )["Installs"].sum().nlargest(10).index

    df5 = df5[
        df5["Category"].isin(top_categories)
    ]

    summary = df5.groupby(
        "Category"
    ).agg({
        "Rating":"mean",
        "Reviews":"sum"
    }).reset_index()

    summary_melt = summary.melt(
        id_vars="Category",
        value_vars=["Rating","Reviews"]
    )

    task_fig5 = px.bar(
        summary_melt,
        x="Category",
        y="value",
        color="variable",
        barmode="group",
        title="Task 5 Average Rating vs Reviews"
    )

    task_fig5.update_layout(
    width=400,
    height=300
)

    task_fig5.show()
    save_plot_as_html(
    task_fig5,
    "Internship_Task_5.html",
    "Task 5 analysis"
)

else:
    plot_containers += """
    <div class="plot-container">
        Available only between 3 PM and 5 PM IST
    </div>
    """

In [65]:
if  is_time_allowed(13,14):

    merged_df["Revenue"] = (
        merged_df["Installs"] *
        merged_df["Price"]
    )

    df6 = merged_df[
        (merged_df["Installs"] > 10000) &
        (merged_df["Revenue"] > 10000) &
        (merged_df["Size_MB"] > 15) &
        (merged_df["Content Rating"] == "Everyone") &
        (merged_df["App"].str.len() <= 30)
    ].copy()

    top3 = df6.groupby(
        "Category"
    )["Installs"].sum().nlargest(3).index

    df6 = df6[
        df6["Category"].isin(top3)
    ]

    summary = df6.groupby(
        ["Category","Type"]
    ).agg({
        "Installs":"mean",
        "Revenue":"mean"
    }).reset_index()

    from plotly.subplots import make_subplots
    import plotly.graph_objects as go

    task_fig6 = make_subplots(
        specs=[[{"secondary_y":True}]]
    )

    for t in summary["Type"].unique():

        d = summary[
            summary["Type"] == t
        ]

        task_fig6.add_trace(
            go.Bar(
                x=d["Category"],
                y=d["Installs"],
                name=f"{t} Installs"
            ),
            secondary_y=False
        )

        task_fig6.add_trace(
            go.Scatter(
                x=d["Category"],
                y=d["Revenue"],
                mode="lines+markers",
                name=f"{t} Revenue"
            ),
            secondary_y=True
        )

    task_fig6.update_layout(
        title="Task 6 Average Installs vs Revenue"
    )

    task_fig6.update_layout(
    width=400,
    height=300
)

    task_fig6.show()
    save_plot_as_html(
    task_fig6,
    "Internship_Task_6.html",
    "Average Installs vs Revenue"
)
    
else:
    plot_containers += """
    <div class="plot-container">
         Available only between 1 PM and 2 PM IST
    </div>
    """

In [66]:
import os

for f in ["task1.html","task2.html","task3.html","task4.html","task5.html","task6.html"]:
    print(f, os.path.exists(f))

task1.html True
task2.html True
task3.html True
task4.html True
task5.html True
task6.html True


In [67]:
plot_containers_split=plot_containers.split('</div>')

In [68]:
if len(plot_containers_split) > 1:
    final_plot=plot_containers_split[-2]+'</div>'
else:
    final_plot=plot_containers

In [69]:
dashboard_html= """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name=viewport" content="width=device-width,initial-scale-1.0">
    <title> Google Play Store Review Analytics</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #333;
            color: #fff;
            margin: 0;
            padding: 0;
        }}
        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
            background-color: #444
        }}
        .header img {{
            margin: 0 10px;
            height: 50px;
        }}
        .container {{
            display: flex;
            flex-wrap: wrap;
            justify_content: center;
            padding: 20px;
        }}
        .plot-container {{
            border: 2px solid #555
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
        }}
        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0,0,0,0.7);
            padding: 5px;
            border-radius: 5px;
            color: #fff;
        }}
        .plot-container: hover .insights {{
            display: block;
        }}
        </style>
        <script>
            function openPlot(filename) {{
                window.open(filename, '_blank');
                }}
        </script>
    </head>
    <body>
        <div class= "header">
            <img src="https://static.vecteezy.com/ti/vecteur-libre/p1/21496372-google-logo-symbole-conception-vecteur-illustration-avec-noir-contexte-gratuit-vectoriel.jpg"
         alt="Google Logo">

            <h1>Google Play Store Reviews Analytics</h1>
            <img src="https://play.google.com/intl/en_us/badges/images/generic/en_badge_web_generic.png?hl=hi"
         alt="Google Play Store Logo">
        </div>
        <div class="container">
            {plots}
        </div>
    </body>
    </html>
    """

In [70]:
final_html = dashboard_html.format(
    plots=plot_containers,
    plot_width=plot_width,
    plot_height=plot_height
)

dashboard_path = os.path.join(
    html_files_path,
    "web page.html"
)

with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)

webbrowser.open(
    'file://' + os.path.realpath(dashboard_path)
)

True